# Insight 4: Instant Lead Qualification (Hot vs Cold)

**Algorithm:** Perceptron, implemented completely from scratch using the
classic perceptron learning rule (Rosenblatt) — no scikit-learn.

**Business use:** at the moment a new lead comes in, instantly classify it
as Hot (route to a senior sales rep now) or Cold (send to an automated
nurture sequence) using just a handful of strong signals.

> Run `1_generate_dataset.ipynb` first so `data/client_data.csv` exists.

## Shared helper functions

Manual implementations of train/test split, feature standardization, and
classification metrics. No scikit-learn is used — everything is plain NumPy.

In [ ]:
import numpy as np
import pandas as pd

def train_test_split(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    n = X.shape[0]
    indices = np.random.permutation(n)
    test_count = int(n * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)
        std[std == 0] = 1.0
    return (X - mean) / std, mean, std

def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision_recall_f1(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def print_classification_report(y_true, y_pred, title="Classification Report"):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    acc = accuracy(y_true, y_pred)
    prec, rec, f1 = precision_recall_f1(y_true, y_pred)
    print(f"\n--- {title} ---")
    print(f"Confusion Matrix:  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

## The model: Perceptron (classic learning rule)

In [ ]:
class PerceptronScratch:
    """
    Classic perceptron: weights update only on misclassified samples,
    labels are treated internally as {-1, +1} as in the original
    Rosenblatt perceptron algorithm.
    """

    def __init__(self, lr=0.01, n_iters=100):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.errors_per_epoch = []

    @staticmethod
    def _activation(z):
        return np.where(z >= 0, 1, -1)

    def fit(self, X, y):
        y_signed = np.where(y == 1, 1, -1)

        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for epoch in range(self.n_iters):
            errors = 0
            for xi, target in zip(X, y_signed):
                z = np.dot(xi, self.weights) + self.bias
                y_pred = self._activation(z)

                if y_pred != target:
                    update = self.lr * target
                    self.weights += update * xi
                    self.bias += update
                    errors += 1

            self.errors_per_epoch.append(errors)

            if errors == 0:
                break  # converged - data is linearly separable

        return self

    def predict(self, X):
        z = X @ self.weights + self.bias
        y_signed = self._activation(z)
        return np.where(y_signed == 1, 1, 0)

## Load data and prepare features

In [ ]:
df = pd.read_csv("data/client_data.csv")

feature_cols = [
    "inquiry_source_score",
    "mentioned_business_income",
    "response_time_hours",
    "requested_pricing_info",
]
target_col = "lead_hot"

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_s, mean, std = standardize(X_train)
X_test_s, _, _ = standardize(X_test, mean, std)

X_train_s.shape, X_test_s.shape

## Train the model

In [ ]:
model = PerceptronScratch(lr=0.01, n_iters=200)
model.fit(X_train_s, y_train)

print("Misclassifications per epoch (first 20 epochs):", model.errors_per_epoch[:20])
print(f"Converged after {len(model.errors_per_epoch)} epoch(s)")

## Evaluate

In [ ]:
y_pred_train = model.predict(X_train_s)
y_pred_test = model.predict(X_test_s)

print_classification_report(y_train, y_pred_train, title="TRAIN performance")
print_classification_report(y_test, y_pred_test, title="TEST performance")

print("\nLearned weights (standardized features):")
for name, w in zip(feature_cols, model.weights):
    print(f"  {name:30s}: {w:8.3f}")
print(f"  {'bias':30s}: {model.bias:8.3f}")

## Business output: sample lead routing decisions

In [ ]:
labels = {1: "HOT -> route to senior rep now", 0: "COLD -> automated nurture email"}
print("Sample lead routing decisions (test set):")
for i in range(5):
    print(f"  Predicted: {labels[y_pred_test[i]]:35s}  Actual label: {int(y_test[i])}")